In [23]:
import os

# Adjust the path to match your Box sync folder location
box_path = os.path.expanduser("/Users/mobinaamrollahi/Library/CloudStorage/Box-Box/Human_AGV_project/ISU_Modeling/Code")

In [24]:
# base libraries
import numpy as np
import pandas as pd
# regular expression module in python to find all sequences of digits in a given string
import re

import os
from datetime import datetime, timedelta
import math
from scipy import stats

In [25]:
Per_Interaction_Data = pd.read_csv(os.path.join(box_path, "Per_Interaction_Data.csv"))
pre_survey = pd.read_csv(os.path.join(box_path, "pre_survey.csv"))
post_survey = pd.read_csv(os.path.join(box_path, "post_survey.csv"))


In [26]:
# Ensure the PID column is treated as a string and add leading zeros
Per_Interaction_Data['PID'] = Per_Interaction_Data['PID'].astype(str).apply(lambda x: x.zfill(3))
pre_survey['PID'] = pre_survey['PID'].astype(str).apply(lambda x: x.zfill(3))
post_survey['PID'] = post_survey['PID'].astype(str).apply(lambda x: x.zfill(3))

In [28]:
# Check the data types of 'DRate' and 'PID' columns in both DataFrames
print("pre_survey data types:")
print(pre_survey[['PID']].dtypes)

print("\nPer_Interaction_Data data types:")
print(Per_Interaction_Data[['DRate', 'PID']].dtypes)

print("\npost_survey data types:")
print(post_survey[['DRate', 'PID']].dtypes)

# If needed, convert the data types to ensure consistency
Per_Interaction_Data['DRate'] = Per_Interaction_Data['DRate'].astype(str)
Per_Interaction_Data['PID'] = Per_Interaction_Data['PID'].astype(str)

post_survey['DRate'] = post_survey['DRate'].astype(str)
post_survey['PID'] = post_survey['PID'].astype(str)

pre_survey data types:
PID    object
dtype: object

Per_Interaction_Data data types:
DRate    object
PID      object
dtype: object

post_survey data types:
DRate    object
PID      object
dtype: object


## Merge Pre, Post, and Study DataFrames based on common columns

In [29]:
# Add new columns with empty strings
new_columns = {
    'Age': "", 'Gender': "", 'Ethnicity': "", 'GamingFrequency': "", 'VRExperience': "", 
    'VRHeadsetExperience': "", 'AGVInteraction': "", 'PerfectAutomation': "", 
    'TrustPropensity': "", 'AutomationExperience': "", 'Trust1': "", 'Trust2': "", 
    'MWL': "", 'Assessment': "",'Frechet_Distance':"",
}

# Add the new columns to the DataFrame
Per_Interaction_Data = Per_Interaction_Data.assign(**new_columns)

In [30]:
Per_Interaction_Data.head()

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,VRHeadsetExperience,AGVInteraction,PerfectAutomation,TrustPropensity,AutomationExperience,Trust1,Trust2,MWL,Assessment,Frechet_Distance
0,001,High,2024-5-3,14,28,53,2,15:2:48,15:3:38,219,...,,,,,,,,,,
1,001,High,2024-5-3,14,28,53,1,15:3:47,15:4:29,0,...,,,,,,,,,,
2,001,High,2024-5-3,14,28,53,3,15:4:37,15:5:34,268,...,,,,,,,,,,
3,001,High,2024-5-3,14,28,53,4,15:5:46,15:6:37,779,...,,,,,,,,,,
4,001,High,2024-5-3,14,28,53,5,15:6:41,15:7:44,868,...,,,,,,,,,,


In [43]:
Per_Interaction_Data.columns

Index(['PID', 'DRate', 'date', 'hr', 'min', 's', 'AGVname', 'StartTime',
       'EndTime', 'GazeDuration', 'mean_dist', 'min_dist', 'max_dist',
       'std_dist', 'mean_agv_spd', 'min_agv_spd', 'max_agv_spd', 'std_spd',
       'task', 'Trust', 'Safe', 'Comfort', 'Expect', 'Age', 'Gender',
       'Ethnicity', 'GamingFrequency', 'VRExperience', 'VRHeadsetExperience',
       'AGVInteraction', 'PerfectAutomation', 'TrustPropensity',
       'AutomationExperience', 'Trust1', 'Trust2', 'MWL', 'Assessment',
       'Frechet_Distance', 'User_Trajectory', 'AGV_Approaching',
       'AGV_User_Combination', 'AGV_Path_Complexity'],
      dtype='object')

### 'Age', 'Gender', 'Ethnicity', 'GamingFrequency', 'VRExperience', 'VRHeadsetExperience', 'AGVInteraction', 'PerfectAutomation', 'TrustPropensity', and 'AutomationExperience' are extracted from the pre-experiment survey.

In [31]:
# Create the dictionary mapping only the desired columns
pre_dict = pre_survey.set_index('PID')[['Age', 'Gender', 'Ethnicity', 'GamingFrequency', 'VRExperience',
                                         'VRHeadsetExperience', 'AGVInteraction', 'PerfectAutomation',
                                         'TrustPropensity', 'AutomationExperience']].to_dict()

# Map the values from pre_survey to Per_Interaction_Data based on the unique 'PID'
for col in pre_dict.keys():
    Per_Interaction_Data[col] = [pre_dict[col].get(pid) for pid in Per_Interaction_Data['PID']]

### 'Trust1', 'Trust2', 'MWL', 'Assessment' are extracted from the post-experiment survey.

In [32]:
# Create the dictionary mapping only the desired columns
post_dict = post_survey.set_index(['DRate', 'PID'])[['Trust1', 'Trust2', 'MWL', 'Assessment']].to_dict()

# Map the values from post_survey to Per_Interaction_Data based on the unique 'PID' and 'DRate'
for col in post_dict.keys():
    Per_Interaction_Data[col] = [post_dict[col].get((drate, pid)) for drate, pid in Per_Interaction_Data[['DRate', 'PID']].values]

In [33]:
Per_Interaction_Data.head()

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,VRHeadsetExperience,AGVInteraction,PerfectAutomation,TrustPropensity,AutomationExperience,Trust1,Trust2,MWL,Assessment,Frechet_Distance
0,001,High,2024-5-3,14,28,53,2,15:2:48,15:3:38,219,...,Moderately experienced,Never,3.571,3.667,3.917,49.0,52.0,63.0,1.812,
1,001,High,2024-5-3,14,28,53,1,15:3:47,15:4:29,0,...,Moderately experienced,Never,3.571,3.667,3.917,49.0,52.0,63.0,1.812,
2,001,High,2024-5-3,14,28,53,3,15:4:37,15:5:34,268,...,Moderately experienced,Never,3.571,3.667,3.917,49.0,52.0,63.0,1.812,
3,001,High,2024-5-3,14,28,53,4,15:5:46,15:6:37,779,...,Moderately experienced,Never,3.571,3.667,3.917,49.0,52.0,63.0,1.812,
4,001,High,2024-5-3,14,28,53,5,15:6:41,15:7:44,868,...,Moderately experienced,Never,3.571,3.667,3.917,49.0,52.0,63.0,1.812,


### Swap the values 1 and 2 in the 'AGVname' column

In [34]:
# Swap the values 1 and 2 in the 'AGVname' column
Per_Interaction_Data.AGVname.replace([2, 1], ['first', 'second'], inplace=True)
Per_Interaction_Data.AGVname.replace(['first', 'second'],[1, 2], inplace=True)

Per_Interaction_Data.head()

/var/folders/sd/mgpkzs5d4ng7v524bgzh394r0000gn/T/ipykernel_35952/3623332704.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  Per_Interaction_Data.AGVname.replace([2, 1], ['first', 'second'], inplace=True)
/var/folders/sd/mgpkzs5d4ng7v524bgzh394r0000gn/T/ipykernel_35952/3623332704.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('futu

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,VRHeadsetExperience,AGVInteraction,PerfectAutomation,TrustPropensity,AutomationExperience,Trust1,Trust2,MWL,Assessment,Frechet_Distance
0,001,High,2024-5-3,14,28,53,1,15:2:48,15:3:38,219,...,Moderately experienced,Never,3.571,3.667,3.917,49.0,52.0,63.0,1.812,
1,001,High,2024-5-3,14,28,53,2,15:3:47,15:4:29,0,...,Moderately experienced,Never,3.571,3.667,3.917,49.0,52.0,63.0,1.812,
2,001,High,2024-5-3,14,28,53,3,15:4:37,15:5:34,268,...,Moderately experienced,Never,3.571,3.667,3.917,49.0,52.0,63.0,1.812,
3,001,High,2024-5-3,14,28,53,4,15:5:46,15:6:37,779,...,Moderately experienced,Never,3.571,3.667,3.917,49.0,52.0,63.0,1.812,
4,001,High,2024-5-3,14,28,53,5,15:6:41,15:7:44,868,...,Moderately experienced,Never,3.571,3.667,3.917,49.0,52.0,63.0,1.812,


In [35]:
# Drop PID = 5, 18, 23, 25. The first three don't have any recorded data, the last one, only has one round of data.
Per_Interaction_Data = Per_Interaction_Data.drop(Per_Interaction_Data[Per_Interaction_Data['PID'].isin([5, 18, 23, 25])].index)

### 'Percentage Looking at AGV' is calculated based on the percentage of 'True' values in the Gaze_on_AGV column for each unique pair of PID, DRate, and AGVname in the ISU_data_by_0.1_sec.

In [ ]:
ISU_data = pd.read_csv(os.path.join(box_path,'ISU_data_by_0.1_sec.csv'))

In [ ]:
# AGVname column in this dataset has AGV[x], and we want to get rid of AGV to be consistent with rest of the datasets
def extract_number(string):
    # Find all sequences of digits in the string
    numbers = re.findall(r'\d+', string)
    # Join the found numbers (if any) an convert to integer
    return int(numbers[0]) if numbers else None 

ISU_data['AGVname'] = ISU_data['AGVname'].apply(extract_number)

# Apply zero-padding to the PID column to be consistent with the corresponding column in Per Interaction dataset
ISU_data['PID'] = ISU_data['PID'].apply(lambda x: str(x).zfill(3))

In [12]:
# Dictionary to store the results
result_dict_for_Gaze_on_AGV = {}

# Grouped by PID, AGVname, and DRate
grouped_for_Gaze = ISU_data.groupby(['PID', 'AGVname', 'DRate'])

for name, group in grouped_for_Gaze:
    # Calculating the faction of True Values with three significant figures
    # When you call sum() method on boolean series, it treats true as 1 and false as 0
    true_count = group['Gaze_on_AGV'].sum()
    total_count = group['Gaze_on_AGV'].count()
    fraction = round(true_count/ total_count, 3)

    # Store the rsults in the dictionary
    result_dict_for_Gaze_on_AGV[name] = fraction

# If you want print a specific key value;
# for key, value in result_dict_for_Gaze_on_AGV.items():
#    if str(key[0]).startswith('003') and key[2] == 'Low':
#        print(f"Key: {key}, Value: {value}")

Per_Interaction_Data['Gaze_on_AGV'] = Per_Interaction_Data.apply(lambda row: result_dict_for_Gaze_on_AGV.get((row['PID'], row['AGVname'], row['DRate']), None), axis=1)

### Calculating 'User_Relative_Speed'

In [19]:
# We'd add three new columns to the ISU_data_by_0.1_sec
ISU_data = ISU_data.assign(User_X_Difference = None, User_Y_Difference = None, User_Distance = None)

ISU_data['User_X_Difference'] = ISU_data.groupby(['PID', 'AGVname', 'DRate'])['User_X'].diff().fillna(0).abs()
ISU_data['User_Y_Difference'] = ISU_data.groupby(['PID', 'AGVname', 'DRate'])['User_Y'].diff().fillna(0).abs()

ISU_data['User_Distance'] = np.sqrt(ISU_data['User_X_Difference']**2 + ISU_data['User_Y_Difference']**2)

# Dictionary to store the results
result_dict_for_User_Relative_Speed = {}

# Grouped by PID, AGVname, and DRate
grouped_for_Speed = ISU_data.groupby(['PID', 'AGVname', 'DRate'])

for name, group in grouped_for_Speed:
    # Calculating the faction of True Values with three significant figures
    # When you call sum() method on boolean series, it treats true as 1 and false as 0
    true_count = group['User_Distance'].sum()
    total_count = group['User_Distance'].count()
    fraction = round(true_count/(total_count*0.1), 3)

    # Store the rsults in the dictionary
    result_dict_for_User_Relative_Speed[name] = fraction

Per_Interaction_Data['User_Relative_Speed'] = Per_Interaction_Data.apply(lambda row: result_dict_for_User_Relative_Speed.get((row['PID'], row['AGVname'], row['DRate']), None), axis=1)

Per_Interaction_Data.to_csv(os.path.join(box_path, 'Per_Interaction_Data_Merged.csv'), index=False)

# Printing the User_X_Difference values for a specific group
#specific_group = ISU_data[
#(ISU_data['PID'] == '002') &
#(ISU_data['AGVname'] == 2) &
#(ISU_data['DRate'] == 'Low')]

#print(specific_group['User_X_Difference'].values)

NameError: name 'ISU_data' is not defined

### Calculating 'Frechet Distance'

In [1]:
import sys
from frechetdist import frdist
from concurrent.futures import ThreadPoolExecutor

ModuleNotFoundError: No module named 'frechetdist'

In [16]:
#Rindi

# Increasing the recursion limit
sys.setrecursionlimit(10000)

# function to calculate Euclidean Distance
def euclidean_distance(p1, p2):
    return np.linalg.norm(np.array(p1) - np.array(p2))

# function to calculate Fréchet Distance
def calculate_frechet_distance(P, Q):
    len_p, len_q = len(P), len(Q)
    ca = np.full((len_p, len_q), -1.0)
    
    def c(i, j):
        if ca[i, j] > -1:
            return ca[i, j]
        elif i == 0 and j == 0:
            ca[i, j] = euclidean_distance(P[0], Q[0])
        elif i > 0 and j == 0:
            ca[i, j] = max(c(i - 1, 0), euclidean_distance(P[i], Q[0]))
        elif i == 0 and j > 0:
            ca[i, j] = max(c(0, j - 1), euclidean_distance(P[0], Q[j]))
        elif i > 0 and j > 0:
            ca[i, j] = max(min(c(i - 1, j), c(i - 1, j - 1), c(i, j - 1)),
                           euclidean_distance(P[i], Q[j]))
        else:
            ca[i, j] = float('inf')
        return ca[i, j]

    for i in range(len_p):
        for j in range(len_q):
            c(i, j)
    
    return ca[len_p - 1, len_q - 1]

# Function to generate expected trajectory points
def generate_expected_trajectory(start, end, num_points):
    return [(start[0], start[1] + (end[1] - start[1]) * i / (num_points - 1)) for i in range(num_points)]

# Function to calculate the Fréchet Distance for each group
def calculate_frechet_distance_for_group(group):
    # Downsampling the group
    group = group.iloc[::10, :]  # number denotes the downsampling interval
    
    # Extracting actual trajectory points
    actual_points = list(zip(group['User_X'], group['User_Y']))
    
    # Defining start and end points
    start_point = actual_points[0]
    end_point = actual_points[-1]
    
    # Generating expected trajectory points
    expected_points = generate_expected_trajectory(start_point, end_point, len(actual_points))
    
    # Calculating Fréchet Distance
    frechet_distance = calculate_frechet_distance(expected_points, actual_points)
    
    return frechet_distance

# Function to process each group and return the result
def process_group(group_info):
    pid, agvname, drate, group = group_info
    frechet_distance = calculate_frechet_distance_for_group(group)
    result = {
        'PID': pid,
        'AGVname': agvname,
        'DRate': drate,
        'Frechet_Distance': frechet_distance
    }
    return result

# Creating an empty DataFrame to store results
results_df = pd.DataFrame(columns=['PID', 'AGVname', 'DRate', 'Frechet_Distance'])

# Processing data for specific PIDs based on user input
while True:
    user_input = input("Enter PID to process (or 'exit' to finish): ")
    if user_input.lower() == 'exit':
        break
    
    try:
        pid = int(user_input)
    except ValueError:
        print("Invalid input. Please enter a valid PID or 'exit' to finish.")
        continue
    
    # Processing each group for the specified PID
    pid_data = ISU_data[ISU_data['PID'] == pid]
    if pid_data.empty:
        print(f"No data found for PID {pid}.")
        continue

    groups = [(pid, agvname, drate, group) for (agvname, drate), group in pid_data.groupby(['AGVname', 'DRate'])]

    with ThreadPoolExecutor() as executor:
        results = list(executor.map(process_group, groups))

    for result in results:
        # Use pd.concat to append the result to results_df
        results_df = pd.concat([results_df, pd.DataFrame([result])], ignore_index=True)
        
        # Print progress
        print(f"Processed PID {result['PID']}, AGVname {result['AGVname']}, DRate {result['DRate']}. Frechet Distance: {result['Frechet_Distance']}")

# Displaying a sample of the results DataFrame to verify the merge
#print(results_df.head())

Enter PID to process (or 'exit' to finish):  2


No data found for PID 2.


Enter PID to process (or 'exit' to finish):  002


No data found for PID 2.


Enter PID to process (or 'exit' to finish):  exit


In [17]:
# Rindi

# Increasing the recursion limit
sys.setrecursionlimit(10000)

# Custom function to calculate Euclidean Distance
def euclidean_distance(p1, p2):
    return np.sqrt((p1[0] - p2[0]) ** 2 + (p1[1] - p2[1]) ** 2)

# Custom iterative function to calculate Fréchet Distance
def calculate_frechet_distance(P, Q):
    len_p, len_q = len(P), len(Q)
    ca = np.full((len_p, len_q), -1.0)
    
    def c(i, j):
        if ca[i, j] > -1:
            return ca[i, j]
        elif i == 0 and j == 0:
            ca[i, j] = euclidean_distance(P[0], Q[0])
        elif i > 0 and j == 0:
            ca[i, j] = max(c(i - 1, 0), euclidean_distance(P[i], Q[0]))
        elif i == 0 and j > 0:
            ca[i, j] = max(c(0, j - 1), euclidean_distance(P[0], Q[j]))
        elif i > 0 and j > 0:
            ca[i, j] = max(min(c(i - 1, j), c(i - 1, j - 1), c(i, j - 1)),
                           euclidean_distance(P[i], Q[j]))
        else:
            ca[i, j] = float('inf')
        return ca[i, j]

    for i in range(len_p):
        for j in range(len_q):
            c(i, j)
    
    return ca[len_p - 1, len_q - 1]

# Function to generate expected trajectory points
def generate_expected_trajectory(start, end, num_points):
    return [(start[0], start[1] + (end[1] - start[1]) * i / (num_points - 1)) for i in range(num_points)]

# Function to calculate the Fréchet Distance for each group
def calculate_frechet_distance_for_group(group):
    # Extracting actual trajectory points
    actual_points = list(zip(group['User_X'], group['User_Y']))
    
    # Defining start and end points
    start_point = (actual_points[0][0], 0)
    end_point = (actual_points[0][0], 10)
    
    # Generating expected trajectory points
    expected_points = generate_expected_trajectory(start_point, end_point, len(actual_points))
    
    # Calculating Fréchet Distance
    frechet_distance = calculate_frechet_distance(expected_points, actual_points)
    
    return frechet_distance

# Creating an empty DataFrame to store results
results_df = pd.DataFrame(columns=['PID', 'AGVname', 'DRate', 'Frechet_Distance'])

# Processing data for specific PIDs based on user input
while True:
    user_input = input("Enter PID to process (or 'exit' to finish): ")
    if user_input.lower() == 'exit':
        break
    
    try:
        pid = int(user_input)
    except ValueError:
        print("Invalid input. Please enter a valid PID or 'exit' to finish.")
        continue
    
    # Processing each group for the specified PID
    pid_data = ISU_data[ISU_data['PID'] == pid]
    if pid_data.empty:
        print(f"No data found for PID {pid}.")
        continue

    for (agvname, drate), group in pid_data.groupby(['AGVname', 'DRate']):
        frechet_distance = calculate_frechet_distance_for_group(group)
        result = {
            'PID': pid,
            'AGVname': agvname,
            'DRate': drate,
            'Frechet_Distance': frechet_distance
        }
        
        # Using pd.concat to append the result to results_df
        results_df = pd.concat([results_df, pd.DataFrame([result])], ignore_index=True)
        
        # Printing progress and result
        print(f"Processed PID {pid}, AGVname {agvname}, DRate {drate}. Frechet_Distance: {frechet_distance}")

# Displaying the results DataFrame 
#print(results_df)

KeyboardInterrupt: 

### 'AGV_Approaching', 'User_Trajectory'

In [38]:
user_trajectory_mapping = {
    2:'Diagonal',
    1:'Straight',
    3:'Diagonal',
    4:'Straight',
    5:'Straight',
    6:'Diagonal',
    7:'Straight',
    8:'Diagonal',
    9:'Diagonal',
    10:'Straight',
    11:'Diagonal',
    12:'Straight',
    13:'Straight',
    14:'Diagonal',
    15:'Straight',
    16:'Diagonal',    
}

Per_Interaction_Data['User_Trajectory'] = Per_Interaction_Data['AGVname'].map(user_trajectory_mapping)

In [39]:
agv_approaching_mapping = {
    2:'North',
    1:'South',
    3:'South',
    4:'Northeast',
    5:'Northwest',
    6:'Northwest',
    7:'East',
    8:'Southwest',
    9:'Northeast',
    10:'West',
    11:'East',
    12:'Southeast',
    13:'Southwest',
    14:'Southeast',
    15:'North',
    16:'West',    
}

Per_Interaction_Data['AGV_Approaching'] = Per_Interaction_Data['AGVname'].map(agv_approaching_mapping)

### Changing AGVname column entries format

In [40]:
Per_Interaction_Data['AGV_User_Combination'] = Per_Interaction_Data['AGV_Approaching'].astype(str) + ' - ' + Per_Interaction_Data['User_Trajectory'].astype(str)

Per_Interaction_Data.iloc[0:17]

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,TrustPropensity,AutomationExperience,Trust1,Trust2,MWL,Assessment,Frechet_Distance,User_Trajectory,AGV_Approaching,AGV_User_Combination
0,001,High,2024-5-3,14,28,53,1,15:2:48,15:3:38,219,...,3.667,3.917,49.0,52.0,63.0,1.812,,Straight,South,South - Straight
1,001,High,2024-5-3,14,28,53,2,15:3:47,15:4:29,0,...,3.667,3.917,49.0,52.0,63.0,1.812,,Diagonal,North,North - Diagonal
2,001,High,2024-5-3,14,28,53,3,15:4:37,15:5:34,268,...,3.667,3.917,49.0,52.0,63.0,1.812,,Diagonal,South,South - Diagonal
3,001,High,2024-5-3,14,28,53,4,15:5:46,15:6:37,779,...,3.667,3.917,49.0,52.0,63.0,1.812,,Straight,Northeast,Northeast - Straight
4,001,High,2024-5-3,14,28,53,5,15:6:41,15:7:44,868,...,3.667,3.917,49.0,52.0,63.0,1.812,,Straight,Northwest,Northwest - Straight
5,001,High,2024-5-3,14,28,53,6,15:7:56,15:8:58,1195,...,3.667,3.917,49.0,52.0,63.0,1.812,,Diagonal,Northwest,Northwest - Diagonal
6,001,High,2024-5-3,14,28,53,7,15:9:7,15:9:50,537,...,3.667,3.917,49.0,52.0,63.0,1.812,,Straight,East,East - Straight
7,001,High,2024-5-3,14,28,53,8,15:10:7,15:11:8,650,...,3.667,3.917,49.0,52.0,63.0,1.812,,Diagonal,Southwest,Southwest - Diagonal
8,001,High,2024-5-3,14,28,53,9,15:11:12,15:12:13,856,...,3.667,3.917,49.0,52.0,63.0,1.812,,Diagonal,Northeast,Northeast - Diagonal
9,001,High,2024-5-3,14,28,53,10,15:12:20,15:13:8,673,...,3.667,3.917,49.0,52.0,63.0,1.812,,Straight,West,West - Straight


### AGV Path Complexity

In [41]:
complexity_mapping = {
    'North - Diagonal': 'Complex',
    'South - Straight': 'Straight',
    'South - Diagonal':'Complex',
    'Northeast - Straight':'Complex',
    'Northwest - Straight':'Complex',
    'Northwest - Diagonal':'Complex',
    'East - Straight':'Straight',
    'Southwest - Diagonal':'Straight',
    'Northeast - Diagonal':'Straight',
    'West - Straight':'Straight',
    'East - Diagonal':'Complex',
    'Southeast - Straight':'Complex',
    'Southwest - Straight':'Complex',
    'Southeast - Diagonal':'Straight',
    'North - Straight':'Straight',
    'West - Diagonal':'Complex',
}

Per_Interaction_Data['AGV_Path_Complexity'] = Per_Interaction_Data['AGV_User_Combination'].map(complexity_mapping)

In [42]:
Per_Interaction_Data.iloc[0:17]

,PID,DRate,date,hr,min,s,AGVname,StartTime,EndTime,GazeDuration,...,AutomationExperience,Trust1,Trust2,MWL,Assessment,Frechet_Distance,User_Trajectory,AGV_Approaching,AGV_User_Combination,AGV_Path_Complexity
0,001,High,2024-5-3,14,28,53,1,15:2:48,15:3:38,219,...,3.917,49.0,52.0,63.0,1.812,,Straight,South,South - Straight,Straight
1,001,High,2024-5-3,14,28,53,2,15:3:47,15:4:29,0,...,3.917,49.0,52.0,63.0,1.812,,Diagonal,North,North - Diagonal,Complex
2,001,High,2024-5-3,14,28,53,3,15:4:37,15:5:34,268,...,3.917,49.0,52.0,63.0,1.812,,Diagonal,South,South - Diagonal,Complex
3,001,High,2024-5-3,14,28,53,4,15:5:46,15:6:37,779,...,3.917,49.0,52.0,63.0,1.812,,Straight,Northeast,Northeast - Straight,Complex
4,001,High,2024-5-3,14,28,53,5,15:6:41,15:7:44,868,...,3.917,49.0,52.0,63.0,1.812,,Straight,Northwest,Northwest - Straight,Complex
5,001,High,2024-5-3,14,28,53,6,15:7:56,15:8:58,1195,...,3.917,49.0,52.0,63.0,1.812,,Diagonal,Northwest,Northwest - Diagonal,Complex
6,001,High,2024-5-3,14,28,53,7,15:9:7,15:9:50,537,...,3.917,49.0,52.0,63.0,1.812,,Straight,East,East - Straight,Straight
7,001,High,2024-5-3,14,28,53,8,15:10:7,15:11:8,650,...,3.917,49.0,52.0,63.0,1.812,,Diagonal,Southwest,Southwest - Diagonal,Straight
8,001,High,2024-5-3,14,28,53,9,15:11:12,15:12:13,856,...,3.917,49.0,52.0,63.0,1.812,,Diagonal,Northeast,Northeast - Diagonal,Straight
9,001,High,2024-5-3,14,28,53,10,15:12:20,15:13:8,673,...,3.917,49.0,52.0,63.0,1.812,,Straight,West,West - Straight,Straight


### 'Cross First' is manually coded using the study video recordings.

In [44]:
# List of columns to analyze
columns_to_analyze = ['mean_dist', 'min_dist', 'max_dist']

# Use describe() to get statistics for the specified columns
stats = Per_Interaction_Data[columns_to_analyze].describe()

# Print the statistics
print("Descriptive Statistics:")
print(stats)

Descriptive Statistics:
         mean_dist     min_dist      max_dist
count  1338.000000  1338.000000   1338.000000
mean   3664.407960   949.699320   6639.170546
std    1355.491928   461.068984   2527.095952
min    1137.970000   130.180000   3128.970000
25%    2711.812500   630.242500   4919.910000
50%    3431.645000   842.075000   5531.010000
75%    4131.142500  1128.757500   7051.935000
max    8890.150000  3046.150000  12733.900000


In [ ]:
# Function to manually input 'Cross First' values
def manually_code_cross_first(row):
    print(f"PID: {row['PID']}, DRate: {row['DRate']}, AGVname: {row['AGVname']}")
    cross_first = input("Enter 'Cross_First' value (Human, AGV[X], Together): ")
    return cross_first

# Ensure the PID column is treated as a string and add leading zeros
Per_Interaction_Data['PID'] = Per_Interaction_Data['PID'].astype(str).apply(lambda x: x.zfill(3))

# Iterate through each row to manually input 'Cross First' values and save after each entry
for index, row in Per_Interaction_Data.iterrows():
    cross_first_value = manually_code_cross_first(row)
    Per_Interaction_Data.at[index, 'Cross_First'] = cross_first_value
    # Save the DataFrame to a CSV file after each entry
    Per_Interaction_Data.to_csv('Per_Interaction_Data_Merged.csv', index=False)

print("All entries have been processed and saved.")

PID: 001, DRate: High, AGVname: 2


Enter 'Cross First' value (Human, AGV[X], Together):  Together


PID: 001, DRate: High, AGVname: 1


Enter 'Cross First' value (Human, AGV[X], Together):  Together


PID: 001, DRate: High, AGVname: 3


...

PID: 001, DRate: High, AGVname: 15


Enter 'Cross First' value (Human, AGV[X], Together):  Together


PID: 001, DRate: High, AGVname: 16


Enter 'Cross First' value (Human, AGV[X], Together):  Together


PID: 001, DRate: Low, AGVname: 2


In [ ]:
Per_Interaction_Data.to_csv(os.path.join(box_path, 'Per_Interaction_Data_Merged.csv'), index=False)